
# Machine Learning aplicado, Clustering 4
## Armar carteras con clustering: de Markowitz a Hierarchical Risk Parity

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos

En los tres notebooks anteriores el clustering agrupó clientes, acciones y hogares. Hoy agrupa activos para una tarea
muy concreta: **decidir cuánto dinero poner en cada uno**. Es la aplicación más directa del clustering a la gestión de
carteras, y para entenderla hay que volver un paso atrás, a cómo se arma una cartera y por qué el método clásico, el
de Markowitz, funciona en la pizarra y falla en la práctica.

La promesa del método que vamos a construir, **Hierarchical Risk Parity** (López de Prado, 2016), es modesta y
honesta: no promete más retorno, promete carteras **estables**, sin posiciones cortas, que no necesitan invertir una
matriz y que aprovechan una estructura que Markowitz ignora: los activos se parecen entre sí por grupos. Y vamos a
terminar con una pregunta incómoda que conviene hacer antes de que la haga un alumno: si repartir el dinero en partes
iguales rinde casi igual que todos los métodos sofisticados, ¿para qué complicarse?

### Qué vas a aprender hoy

1. Repaso de carteras: retorno y riesgo de una combinación de activos, por qué diversificar funciona, y qué dice Markowitz.
2. Las reglas de asignación clásicas, de la más simple a la más ambiciosa: 1/N, inversa de la varianza, mínima varianza, paridad de riesgo y máximo Sharpe.
3. Por qué el optimizador amplifica los errores de estimación, con el ejemplo del máximo Sharpe estallando.
4. HRP paso a paso: de correlaciones a distancias, el dendrograma, la cuasi-diagonalización y la bisección recursiva.
5. Un backtest honesto con 49 industrias, donde 1/N casi gana, y por qué; y un segundo universo con bonos, oro y acciones donde 1/N concentra el riesgo y los métodos basados en riesgo sí importan.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import warnings, os, io, zipfile, urllib.request, time

from scipy.spatial.distance import squareform, pdist
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)



## 1. Repaso: qué es una cartera y qué dice Markowitz

Una cartera es un vector de pesos $w = (w_1, \dots, w_N)$ que suman 1: la fracción del dinero en cada activo. Si los
activos tienen retornos esperados $\mu$ y matriz de covarianzas $\Sigma$, la cartera tiene

$$\text{retorno esperado} = w^\top \mu, \qquad \text{varianza} = w^\top \Sigma\, w = \sum_i \sum_j w_i w_j \sigma_{ij}.$$

La segunda fórmula es la que explica la diversificación. La varianza de la cartera no es el promedio de las varianzas:
depende de las **covarianzas** $\sigma_{ij}$ entre cada par. Si dos activos no se mueven juntos, sus movimientos se
cancelan en parte y la cartera es menos riesgosa que cualquiera de los dos. Con muchos activos poco correlacionados,
el riesgo propio de cada uno casi desaparece y queda solo lo que tienen en común.

Markowitz (1952) convirtió eso en un problema de optimización: para cada nivel de retorno esperado, la cartera de
mínima varianza; el conjunto de esas carteras es la **frontera eficiente**, y la mejor de todas (máximo retorno por
unidad de riesgo) es la cartera de **máximo Sharpe**. Es correcto, es elegante, y tiene un problema que no está en la
matemática sino en los insumos: **nunca conocemos $\mu$ ni $\Sigma$**. Los estimamos con datos históricos, y el
optimizador trata cada error de estimación como información: si un activo tuvo por azar un retorno promedio alto en la
muestra, el optimizador le carga la cartera. Un retorno promedio se estima muy mal (el error estándar de un promedio
diario con un año de datos es del tamaño del promedio mismo), y por eso las carteras "óptimas" cambian por completo de
un mes al otro y rinden peor que reglas tontas. Es el resultado de DeMiguel, Garlappi y Uppal (2009): sobre decenas
de bases de datos, ninguna regla sofisticada le ganó de forma consistente a repartir en partes iguales.

### Las reglas de asignación, de la más simple a la más ambiciosa

| regla | pesos | qué necesita estimar |
|---|---|---|
| **1/N** (equiponderada) | $w_i = 1/N$ | nada |
| **inversa de la varianza** | $w_i \propto 1/\sigma_i^2$ | solo las varianzas |
| **mínima varianza** | $w \propto \Sigma^{-1} \mathbf{1}$ | toda la matriz $\Sigma$, y hay que invertirla |
| **paridad de riesgo** | cada activo aporta la misma fracción de la varianza total | toda la matriz $\Sigma$ |
| **máximo Sharpe** | $w \propto \Sigma^{-1} \mu$ | $\Sigma$ y además $\mu$ |

Mientras más abajo en la tabla, más se puede ganar si los insumos fueran perfectos y más se pierde cuando no lo son.
La paridad de riesgo merece una línea: la contribución al riesgo del activo $i$ es $w_i (\Sigma w)_i / (w^\top \Sigma w)$,
y la regla iguala esas contribuciones; con activos poco correlacionados se parece a la inversa de la varianza, con
activos correlacionados no.


In [ ]:

def equiponderada(ret):
    return pd.Series(1 / ret.shape[1], index=ret.columns)

def inversa_varianza(ret):
    w = 1 / ret.var()
    return w / w.sum()

def minima_varianza(ret):                      # sin restricciones: admite posiciones cortas
    cov = ret.cov().values
    w = np.linalg.pinv(cov) @ np.ones(len(cov))
    return pd.Series(w / w.sum(), index=ret.columns)

def minima_varianza_largo(ret):                # solo posiciones largas, por optimización numérica
    cov = ret.cov().values * 1e4; n = len(cov)
    r = minimize(lambda w: w @ cov @ w, np.ones(n) / n, jac=lambda w: 2 * cov @ w, bounds=[(0, 1)] * n,
                 constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(n)}], method="SLSQP", options={"maxiter": 1000, "ftol": 1e-12})
    return pd.Series(r.x, index=ret.columns)

def paridad_de_riesgo(ret):                    # iguala las contribuciones al riesgo, solo largos
    cov = ret.cov().values * 1e4; n = len(cov)
    def desigualdad(w):
        rc = w * (cov @ w)
        return ((rc - rc.mean()) ** 2).sum()
    w0 = 1 / np.sqrt(np.diag(cov)); w0 = w0 / w0.sum()
    r = minimize(desigualdad, w0, bounds=[(1e-6, 1)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP", options={"maxiter": 1000, "ftol": 1e-14})
    return pd.Series(r.x / r.x.sum(), index=ret.columns)

def maximo_sharpe(ret):                        # la cartera tangente de Markowitz, con promedios muestrales
    cov, mu = ret.cov().values, ret.mean().values
    w = np.linalg.pinv(cov) @ mu
    return pd.Series(w / w.sum(), index=ret.columns)

def contribucion_al_riesgo(w, cov):
    w = w.values; v = w @ cov @ w
    return pd.Series(w * (cov @ w) / v, index=cov.index if hasattr(cov, "index") else None)



## 2. Los datos: 49 industrias

**49 portafolios industriales** de la biblioteca de Kenneth French: cada serie es el retorno diario de una cartera con
todas las empresas estadounidenses de una industria, desde 2000. Es un universo cómodo para aprender porque cubre la
economía completa con pocos activos, sin huecos ni empresas que desaparecen. Más adelante usamos un segundo universo
muy distinto.


In [ ]:

URL_DATOS = ""     # <-- link al archivo del curso (49_Industry_Portfolios_Daily.csv) si lo tienes; si no, se baja de Dartmouth
BASE_FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

def leer_french(texto, bloque=0):
    # los archivos de French traen varios bloques (ponderado por valor, equiponderado...) separados por encabezados de texto
    lineas = texto.splitlines()
    inicios = [i for i, l in enumerate(lineas) if l.startswith(",") and i + 1 < len(lineas) and lineas[i + 1][:8].strip().isdigit()]
    ini = inicios[bloque]
    fin = next(j for j in range(ini + 1, len(lineas) + 1) if j == len(lineas) or not lineas[j][:8].strip().isdigit())
    d = pd.read_csv(io.StringIO("\n".join(lineas[ini:fin])), index_col=0)
    d.index = pd.to_datetime(d.index, format="%Y%m%d"); d.columns = [c.strip() for c in d.columns]
    return d.replace([-99.99, -999], np.nan) / 100

if URL_DATOS:
    texto = urllib.request.urlopen(normalizar_link(URL_DATOS)).read().decode("latin-1")
elif os.path.exists("49_Industry_Portfolios_Daily.csv"):
    texto = open("49_Industry_Portfolios_Daily.csv", encoding="latin-1").read()
else:
    peticion = urllib.request.Request(BASE_FRENCH + "49_Industry_Portfolios_daily_CSV.zip", headers={"User-Agent": "Mozilla/5.0"})
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(peticion, timeout=180).read())) as z:
        texto = z.read(z.namelist()[0]).decode("latin-1")
R = leer_french(texto).loc["2000":].dropna(axis=1).dropna()
vol_anual = R.std() * np.sqrt(252)
print(f"{len(R):,} días, {R.shape[1]} industrias, de {R.index.min().date()} a {R.index.max().date()}")
print(f"Volatilidad anual: mínima {vol_anual.min():.0%} ({vol_anual.idxmin()}), mediana {vol_anual.median():.0%}, máxima {vol_anual.max():.0%} ({vol_anual.idxmax()})")
print("Correlación promedio entre industrias:", round(R.corr().values[np.triu_indices(R.shape[1], 1)].mean(), 2))



## 3. El problema de estimación, visto

Antes de cualquier método nuevo, miremos el problema con los propios datos. Estimamos $\mu$ y $\Sigma$ con un año (252
días) de retornos, calculamos los pesos de cada regla, y repetimos con el año siguiente. Si los insumos estuvieran
bien estimados, los pesos deberían parecerse de un año al otro.


In [ ]:

REGLAS = {"1/N": equiponderada, "inversa de varianza": inversa_varianza, "mínima varianza (sin restricción)": minima_varianza,
          "mínima varianza (solo largos)": minima_varianza_largo, "paridad de riesgo": paridad_de_riesgo, "máximo Sharpe": maximo_sharpe}
a1, a2 = R.loc["2022"], R.loc["2023"]
filas = []
for nombre, f in REGLAS.items():
    w1, w2 = f(a1), f(a2)
    filas.append(dict(regla=nombre, peso_maximo_2022=w1.max(), peso_minimo_2022=w1.min(), cortos_2022=int((w1 < 0).sum()),
                      activos_efectivos_2022=1 / (w1 ** 2).sum(), rotacion_2022_a_2023=np.abs(w1 - w2).sum() / 2))
print("Pesos estimados con 2022 y cuánto cambian al reestimar con 2023 (rotación = fracción de la cartera que hay que transar):")
print(pd.DataFrame(filas).set_index("regla").round(3).to_string())



La fila del máximo Sharpe es el problema de Markowitz en una línea: pesos de varias veces el capital, decenas de
posiciones cortas, y al reestimar un año después hay que dar vuelta la cartera entera. La mínima varianza sin
restricción también toma cortos y rota mucho, aunque menos, porque no usa $\mu$ (que es lo peor estimado) sino solo
$\Sigma$. Las reglas que solo miran varianzas, o que no miran nada, casi no se mueven. Hay una medida que vamos a usar
todo el notebook: el **número efectivo de activos**, $1 / \sum_i w_i^2$, que vale $N$ cuando los pesos son iguales y 1
cuando todo está en un activo.



## 4. Hierarchical Risk Parity, paso a paso

La observación de López de Prado es que el optimizador trata a los 49 activos como si fueran comparables entre sí: al
invertir $\Sigma$ permite que cualquier activo compense a cualquier otro. Pero los activos tienen **estructura de
grupos**: los bancos se parecen entre sí, bastante a las aseguradoras, poco a las mineras y nada al oro. HRP usa esa
estructura en tres pasos: descubre el árbol de parecidos con clustering jerárquico, ordena los activos según el árbol,
y reparte el capital recorriendo el árbol de arriba hacia abajo. Nunca invierte una matriz.

### 4.1 De correlaciones a distancias

El clustering necesita distancias, y la correlación no lo es (va de −1 a 1, y dos activos idénticos tendrían
"distancia" 1). La transformación de Mantegna (1999) sí es una distancia válida:

$$d_{ij} = \sqrt{\tfrac{1}{2}\,(1 - \rho_{ij})}$$

vale 0 si $\rho = 1$, 1 si $\rho = -1$, y cumple la desigualdad triangular. HRP agrega un paso: en vez de agrupar
por $d_{ij}$, agrupa por la distancia euclidiana entre las **columnas** de la matriz $d$, es decir, compara cómo se
relaciona cada activo con **todo** el universo y no solo con el otro.


In [ ]:

def distancia_mantegna(corr):
    return np.sqrt(np.clip(0.5 * (1 - corr), 0, None))

def distancia_de_distancias(D):
    return squareform(pdist(D, metric="euclidean"))

corr = R.corr()
D = distancia_mantegna(corr.values)
D2 = distancia_de_distancias(D)
iu = np.triu_indices_from(D, 1)
pares = pd.DataFrame({"a": np.array(R.columns)[iu[0]], "b": np.array(R.columns)[iu[1]], "distancia": D[iu], "correlacion": corr.values[iu]})
print("Las cinco industrias más parecidas:"); print(pares.nsmallest(5, "distancia").round(3).to_string(index=False))
print(); print("Las cinco más distintas:"); print(pares.nlargest(5, "distancia").round(3).to_string(index=False))



### 4.2 El dendrograma

Con la matriz de distancias construimos un **árbol jerárquico**: cada industria empieza sola, se unen las dos más
cercanas, y se repite hasta que queda un solo grupo. La altura de cada unión es cuán distintos eran los grupos que se
fusionaron. Hace falta decidir cómo se mide la distancia entre dos grupos ya formados (el **criterio de enlace**):
`single` usa el par más cercano, `complete` el más lejano, `average` el promedio y `ward` el que menos aumenta la
varianza. HRP original usa `single`; el ejercicio 2 compara.


In [ ]:

Z_single = linkage(squareform(D2, checks=False), method="single")
fig = ff.create_dendrogram(D2, labels=list(R.columns), distfun=lambda x: squareform(D2, checks=False), linkagefun=lambda x: Z_single, color_threshold=0.7 * Z_single[:, 2].max())
fig.update_layout(title="Dendrograma de las 49 industrias (enlace single, distancia de distancias)", height=480, width=1000, yaxis_title="distancia")
fig.update_xaxes(tickangle=90)
fig.show()

SECTOR = {"Banks": "financiero", "Insur": "financiero", "Fin": "financiero", "RlEst": "financiero", "Hardw": "tecnología", "Softw": "tecnología", "Chips": "tecnología", "LabEq": "tecnología",
          "Oil": "energía y materiales", "Coal": "energía y materiales", "Mines": "energía y materiales", "Gold": "energía y materiales", "Chems": "energía y materiales", "Steel": "energía y materiales",
          "Food": "consumo básico", "Soda": "consumo básico", "Beer": "consumo básico", "Smoke": "consumo básico", "Hshld": "consumo básico", "Util": "consumo básico",
          "Drugs": "salud", "MedEq": "salud", "Hlth": "salud", "Rtail": "consumo discrecional", "Meals": "consumo discrecional", "Clths": "consumo discrecional", "Toys": "consumo discrecional", "Fun": "consumo discrecional", "Autos": "consumo discrecional"}
Z_ward = linkage(squareform(D2, checks=False), method="ward")
grupos = fcluster(Z_ward, 6, criterion="maxclust")
for g in range(1, 7):
    miembros = [c for c, k in zip(R.columns, grupos) if k == g]
    print(f"Grupo {g} ({len(miembros):2d}): " + ", ".join(miembros))
print()
print("Correlación del oro con las demás industrias: promedio", round(corr["Gold"].drop("Gold").mean(), 2), "(el promedio general es", round(corr.values[iu].mean(), 2), ")")



El algoritmo nunca vio la palabra "banco" ni "tecnología", y aun así los grupos son reconocibles: consumo básico y
utilities juntos, recursos naturales juntos, las industrias tecnológicas juntas. Dos detalles valen la pena. El **oro
queda solo**: su correlación con el resto es la más baja de la matriz, y eso lo vuelve el único diversificador genuino
del conjunto. Y hay un grupo grande que mezcla bancos, construcción, maquinaria, transporte y retail: no es un
fracaso del algoritmo, es que esas industrias comparten casi todo su riesgo con el mercado y, desde el punto de vista
de una cartera, son **una sola apuesta cíclica** aunque los sectores declarados sean distintos.

### 4.3 Cuasi-diagonalización: ordenar para ver

Una matriz de correlaciones de 49 activos en orden alfabético no muestra nada. Ordenada según el dendrograma, para
que los activos parecidos queden contiguos, los bloques saltan a la vista. Ese reordenamiento se llama
cuasi-diagonalización y es el segundo paso de HRP: la información queda concentrada cerca de la diagonal.


In [ ]:

def orden_cuasi_diagonal(Z):
    # recorre el árbol y devuelve las hojas en el orden en que aparecen (López de Prado, 2016)
    Z = Z.astype(int)
    orden = pd.Series([Z[-1, 0], Z[-1, 1]]); n_items = Z[-1, 3]
    while orden.max() >= n_items:
        orden.index = range(0, orden.shape[0] * 2, 2)
        df = orden[orden >= n_items]
        i, j = df.index, df.values - n_items
        orden[i] = Z[j, 0]
        orden = pd.concat([orden, pd.Series(Z[j, 1], index=i + 1)]).sort_index()
        orden.index = range(orden.shape[0])
    return orden.tolist()

orden = orden_cuasi_diagonal(Z_single)
etiquetas_ord = [R.columns[i] for i in orden]
fig = go.Figure()
fig.add_trace(go.Heatmap(z=corr.values, x=list(R.columns), y=list(R.columns), zmin=0, zmax=1, colorscale="RdYlBu_r", showscale=False, xaxis="x", yaxis="y"))
fig.add_trace(go.Heatmap(z=corr.iloc[orden, orden].values, x=etiquetas_ord, y=etiquetas_ord, zmin=0, zmax=1, colorscale="RdYlBu_r", xaxis="x2", yaxis="y2"))
fig.update_layout(title="Correlaciones en orden alfabético (izquierda) y cuasi-diagonalizadas por el dendrograma (derecha)", height=520, width=1000,
                  xaxis=dict(domain=[0, 0.47], showticklabels=False), yaxis=dict(showticklabels=False, autorange="reversed"), xaxis2=dict(domain=[0.53, 1], showticklabels=False), yaxis2=dict(anchor="x2", showticklabels=False, autorange="reversed"))
fig.show()
print("Orden que produce el árbol (primeras 20):", ", ".join(etiquetas_ord[:20]))



### 4.4 Bisección recursiva

El tercer paso reparte el capital recorriendo el orden del árbol:

1. Todos los activos en un solo grupo, con peso 1.
2. Partir el grupo en dos mitades según el orden cuasi-diagonal.
3. Calcular la varianza de cada mitad como si dentro de ella los pesos fueran inversos a la varianza.
4. Repartir el peso del grupo entre las dos mitades en proporción **inversa** a su varianza: la mitad más riesgosa recibe menos.
5. Repetir sobre cada mitad hasta llegar a activos individuales.

Como solo multiplica por factores entre 0 y 1, **nunca produce un peso negativo**, y como reparte primero entre
bloques grandes, ningún activo puede acaparar. Y nunca invierte $\Sigma$: todas las decisiones son comparaciones entre
dos bloques, así que el método se puede calcular incluso con más activos que observaciones, donde Markowitz ni siquiera
existe.


In [ ]:

def varianza_de_grupo(cov, items):
    sub = cov.loc[items, items]
    w = 1 / np.diag(sub.values); w = w / w.sum()
    return float(w @ sub.values @ w)

def biseccion_recursiva(cov, orden_items):
    w = pd.Series(1.0, index=orden_items)
    grupos = [orden_items]
    while grupos:
        grupos = [g[a:b] for g in grupos for a, b in ((0, len(g) // 2), (len(g) // 2, len(g))) if len(g) > 1]
        for i in range(0, len(grupos), 2):
            izq, der = grupos[i], grupos[i + 1]
            v_izq, v_der = varianza_de_grupo(cov, izq), varianza_de_grupo(cov, der)
            alfa = 1 - v_izq / (v_izq + v_der)
            w[izq] *= alfa; w[der] *= 1 - alfa
    return w

def hrp(ret, criterio="single"):
    cov, c = ret.cov(), ret.corr()
    d2 = distancia_de_distancias(distancia_mantegna(c.values))
    Z = linkage(squareform(d2, checks=False), method=criterio)
    orden = [c.columns[i] for i in orden_cuasi_diagonal(Z)]
    return biseccion_recursiva(cov, orden).reindex(ret.columns)

REGLAS["HRP"] = hrp
w_hrp = hrp(R)
print(f"Pesos HRP sobre toda la muestra: suman {w_hrp.sum():.3f}, mínimo {w_hrp.min():.3%}, máximo {w_hrp.max():.3%}, activos efectivos {1 / (w_hrp ** 2).sum():.1f} de 49")
pesos = pd.DataFrame({n: f(R) for n, f in REGLAS.items() if n != "máximo Sharpe"}).loc[etiquetas_ord]
fig = px.bar(pesos.reset_index(names="industria").melt(id_vars="industria", var_name="regla", value_name="peso"), x="industria", y="peso", color="regla", barmode="group",
             title="Pesos de cada regla sobre toda la muestra (industrias en el orden del dendrograma)")
fig.update_layout(height=440, width=1000)
fig.show()



## 5. La prueba honesta: fuera de muestra, con ventana móvil

El protocolo estándar: en cada fecha de decisión se estima $\Sigma$ (y $\mu$ para el máximo Sharpe) con los últimos
$T = 252$ días, cada regla produce sus pesos, y esos pesos se mantienen 21 días hábiles; después se avanza y se
reestima. Ninguna decisión usa datos posteriores a su fecha. Medimos retorno, volatilidad, Sharpe, caída máxima y
**rotación** (qué fracción de la cartera hay que transar en cada reestimación, que es un costo real).


In [ ]:

def backtest(R, reglas, T=252, paso=21):
    acum = {m: [] for m in reglas}; rot = {m: [] for m in reglas}; previo = {m: None for m in reglas}; fechas = []
    V = R.values
    for t in range(T, len(R) - paso, paso):
        ventana, futuro = R.iloc[t - T:t], V[t:t + paso]
        for m, f in reglas.items():
            w = f(ventana)
            acum[m].append(futuro @ w.values)
            if previo[m] is not None:
                rot[m].append(np.abs(w.values - previo[m]).sum() / 2)
            previo[m] = w.values
        fechas.append(R.index[t:t + paso])
    carteras = pd.DataFrame({m: np.concatenate(v) for m, v in acum.items()}, index=np.concatenate(fechas))
    resumen = pd.DataFrame({m: dict(retorno=carteras[m].mean() * 252, volatilidad=carteras[m].std() * np.sqrt(252), sharpe=carteras[m].mean() / carteras[m].std() * np.sqrt(252),
                                    caida_maxima=((1 + carteras[m]).cumprod() / (1 + carteras[m]).cumprod().cummax() - 1).min(), rotacion_mensual=np.mean(rot[m])) for m in reglas}).T
    return resumen, carteras

t0 = time.time()
resumen49, carteras49 = backtest(R, REGLAS)
print(f"49 industrias, 2001 a {R.index.max().year}, ventana de 252 días, reestimación mensual   [{time.time() - t0:.0f} s]")
print(resumen49.round(3).to_string())


In [ ]:

acum = (1 + carteras49.drop(columns="máximo Sharpe")).cumprod()
fig = go.Figure()
for c in acum.columns:
    fig.add_scatter(x=acum.index, y=acum[c], name=c, line=dict(width=3 if c in ("1/N", "HRP") else 1.3))
fig.update_layout(title="Capital acumulado fuera de muestra, 49 industrias (el máximo Sharpe queda fuera del gráfico: estalla)", height=460, width=900, yaxis_type="log", yaxis_title="capital (escala log)")
fig.show()



### Y entonces, ¿para qué complicarse?

Léase la tabla sin prejuicios. **El máximo Sharpe es un desastre** (volatilidad de cientos por ciento y una rotación
de decenas de veces la cartera al mes): es lo que pasa cuando el optimizador come promedios muestrales. **Todo lo
demás rinde parecido**, y 1/N, que no estima nada, tiene un Sharpe comparable con el de cualquiera. Los métodos
basados en riesgo bajan la volatilidad (la mínima varianza sin restricción es la que más, a costa de cortos y de
rotar dos tercios de la cartera cada mes; HRP la baja con una rotación cinco veces menor) y la caída máxima, pero la
mejora en Sharpe va de centésimas a una décima a lo más (la mejor es la mínima varianza solo con largos, 0,78 contra
0,62), y el ejercicio 1 muestra cuánto de eso sobrevive a los costos.

Hay dos razones, y las dos importan para saber cuándo usar cada cosa.

**Primera: en este universo los activos son parecidos.** Las 49 industrias tienen volatilidades entre 16% y 50%, con
la mitad entre 21% y 29%, y correlaciones promedio de 0,56. Cuando todos los activos tienen riesgo similar y se mueven
juntos, repartir en partes iguales ya es casi paridad de riesgo, y no hay mucho que un método basado en $\Sigma$ pueda
mejorar. DeMiguel y sus coautores concluyen que para que una regla estimada le gane a 1/N hacen falta ventanas de
estimación de décadas, porque la ganancia teórica es chica y el error de estimación la consume.

**Segunda: la ventaja de los métodos basados en riesgo no está en el Sharpe sino en el perfil.** Menos volatilidad,
menos caída, sin cortos, y en el caso de HRP poca rotación. Para un mandato real con costos de transacción,
restricciones de posición y un comité que pregunta por la caída máxima, ese perfil vale más que unas centésimas de
Sharpe.

Lo que hace falta para que 1/N deje de ser suficiente es un universo donde los activos **no** se parezcan. Eso es lo
que viene.



## 6. Donde 1/N falla: un universo de activos distintos

Catorce ETF que cubren acciones (SPY, QQQ, IWM, EFA, EEM), bonos de gobierno (IEF, TLT), bonos corporativos (LQD,
HYG), inflación (TIP), oro, plata, materias primas y bienes raíces (GLD, SLV, DBC, VNQ), diarios desde 2007. Las
volatilidades van de 6% (TIP) a 31% (SLV): aquí 1/N pone un catorceavo del dinero en cada uno, pero el **riesgo** lo
ponen los más volátiles.


In [ ]:

URL_ETF = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/etf_precios.csv"
try:
    precios_etf = pd.read_csv(URL_ETF, index_col=0, parse_dates=True)
except Exception:
    import yfinance as yf
    precios_etf = yf.download(["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "IEF", "LQD", "HYG", "GLD", "SLV", "DBC", "VNQ", "TIP"], start="2007-01-01", end="2025-12-31", auto_adjust=True, progress=False)["Close"].dropna()
precios_etf = precios_etf.drop(columns=[c for c in ["SHY"] if c in precios_etf.columns])
E = precios_etf.pct_change().dropna()
print(f"{len(E):,} días, {E.shape[1]} ETF, de {E.index.min().date()} a {E.index.max().date()}")
print("Volatilidad anual:", (E.std() * np.sqrt(252)).sort_values().round(2).to_dict())

cov_e = E.cov()
contrib = pd.DataFrame({n: contribucion_al_riesgo(f(E), cov_e) for n, f in REGLAS.items() if n in ("1/N", "paridad de riesgo", "HRP")})
ACCIONES = ["SPY", "QQQ", "IWM", "EFA", "EEM"]
print()
print("Fracción del riesgo total que viene de los cinco ETF de acciones (sobre toda la muestra):")
print(contrib.loc[ACCIONES].sum().round(2).to_string())
fig = px.bar(contrib.reset_index(names="ETF").melt(id_vars="ETF", var_name="regla", value_name="contribución al riesgo"), x="ETF", y="contribución al riesgo", color="regla", barmode="group",
             title="Quién aporta el riesgo en cada cartera: 1/N reparte el dinero, no el riesgo")
fig.update_layout(height=420, width=900, yaxis_tickformat=".0%")
fig.show()


In [ ]:

t0 = time.time()
resumen_etf, carteras_etf = backtest(E, REGLAS)
print(f"14 ETF, 2008 a {E.index.max().year}, ventana de 252 días, reestimación mensual   [{time.time() - t0:.0f} s]")
print(resumen_etf.round(3).to_string())
acum = (1 + carteras_etf.drop(columns="máximo Sharpe")).cumprod()
fig = go.Figure()
for c in acum.columns:
    fig.add_scatter(x=acum.index, y=acum[c], name=c, line=dict(width=3 if c in ("1/N", "HRP") else 1.3))
fig.update_layout(title="Capital acumulado fuera de muestra, 14 ETF", height=460, width=900, yaxis_title="capital")
fig.show()



Aquí la historia cambia. En 1/N, cinco ETF de acciones aportan cerca del 60% del riesgo de una cartera de catorce
activos: la "diversificación" es nominal. Las reglas basadas en riesgo bajan la volatilidad y la caída máxima a la
mitad o menos, con un Sharpe parecido (mejor para la paridad de riesgo y la inversa de varianza, algo menor para HRP),
y HRP lo hace con pesos todos positivos y sin invertir ninguna matriz. El
precio es un retorno absoluto menor (la cartera queda cargada a bonos); quien quiera el mismo retorno que 1/N con menos
riesgo tendría que apalancar la cartera de riesgo, que es exactamente lo que hacen los fondos de paridad de riesgo.

La regla práctica que sale de los dos universos: **1/N es el punto de partida obligatorio** y el benchmark que toda
regla tiene que superar; los métodos basados en riesgo (inversa de varianza, paridad de riesgo, HRP) pagan cuando los
activos tienen riesgos muy distintos o cuando hay muchos activos respecto de los datos; y los métodos que usan
retornos esperados estimados no pagan nunca con promedios muestrales. HRP, en particular, es el que mejor se porta
cuando $N$ es grande y $T$ chico, y el ejercicio 3 lo muestra.

## 7. Lo que hay que llevarse

1. El riesgo de una cartera depende de las covarianzas, no de las varianzas: diversificar es juntar activos que no se mueven juntos.
2. Markowitz es correcto con insumos verdaderos y frágil con insumos estimados; los retornos esperados son lo peor estimado, y el máximo Sharpe muestral es inusable.
3. HRP: distancia de Mantegna, dendrograma, cuasi-diagonalización, bisección recursiva. Sin inversión de matrices, sin cortos, estable.
4. Con activos parecidos, 1/N rinde casi igual que todo; con activos distintos, 1/N concentra el riesgo en los volátiles y los métodos basados en riesgo cortan la volatilidad a la mitad.
5. La ventaja de HRP se mide en volatilidad, caída máxima y rotación, no en Sharpe; y en que funciona cuando $N$ se acerca a $T$.



## Ejercicios

1. **Costos de transacción.** Descuenta del retorno mensual de cada regla un costo proporcional a su rotación: 10, 30
   y 50 puntos base por unidad rotada. Recalcula el Sharpe neto de cada regla en los dos universos. ¿A partir de qué
   costo la mínima varianza sin restricción deja de ser la de menor volatilidad por unidad de costo, y qué pasa con HRP?
2. **El criterio de enlace.** Repite el backtest de las 49 industrias con HRP usando `single`, `complete`, `average` y
   `ward`, y compara volatilidad, Sharpe y rotación. ¿Cambia tanto como la elección de regla?
3. **Cuando N se acerca a T.** Repite el backtest de las 49 industrias con ventanas de 60, 120, 252, 504 y 1.000 días
   solo para 1/N, mínima varianza sin restricción y HRP, y grafica la volatilidad fuera de muestra contra $q = N/T$.
   ¿Qué le pasa a la mínima varianza cuando $q$ se acerca a 1, y qué le pasa a HRP?



## Soluciones


In [ ]:

# Ejercicio 1: costos proporcionales a la rotación
def sharpe_neto(resumen, costo_pb):
    neto = resumen["retorno"] - resumen["rotacion_mensual"] * 12 * costo_pb / 1e4
    return neto / resumen["volatilidad"]
filas = {}
for nombre, res in [("49 industrias", resumen49), ("14 ETF", resumen_etf)]:
    for pb in [0, 10, 30, 50]:
        filas[(nombre, f"{pb} pb")] = sharpe_neto(res, pb)
print("Sharpe neto de costos (el máximo Sharpe se omite):")
print(pd.DataFrame(filas).drop(index="máximo Sharpe").round(3).to_string())



Con costos, la mínima varianza sin restricción pierde su ventaja rápido (rota dos tercios de la cartera cada mes), y
las reglas que casi no rotan (1/N, inversa de varianza, paridad de riesgo) quedan arriba, y HRP, que rota algo más,
pierde unas centésimas. En el universo de ETF la paridad de riesgo y la inversa de varianza se mantienen por sobre 1/N
a cualquier costo razonable; HRP queda un poco por debajo en Sharpe, pero con la mitad de la volatilidad y de la
caída máxima, que es lo que un mandato con límite de riesgo mira primero.


In [ ]:

# Ejercicio 2: criterios de enlace
t0 = time.time()
reglas_enlace = {f"HRP {c}": (lambda ret, c=c: hrp(ret, criterio=c)) for c in ["single", "complete", "average", "ward"]}
res_enlace, _ = backtest(R, reglas_enlace)
print(f"[{time.time() - t0:.0f} s]")
print(res_enlace.round(3).to_string())



Los cuatro criterios dan resultados muy parecidos: el enlace cambia la forma del árbol, pero la bisección recursiva
reparte por varianzas y termina en carteras similares. La decisión que importa es usar HRP o no, no cómo se enlaza.


In [ ]:

# Ejercicio 3: ventana de estimación y q = N/T
t0 = time.time()
filas = []
for T in [60, 120, 252, 504, 1000]:
    res, _ = backtest(R, {k: REGLAS[k] for k in ["1/N", "mínima varianza (sin restricción)", "HRP"]}, T=T)
    for m in res.index:
        filas.append(dict(T=T, q=R.shape[1] / T, regla=m, volatilidad=res.loc[m, "volatilidad"], rotacion=res.loc[m, "rotacion_mensual"]))
sens = pd.DataFrame(filas)
print(f"[{time.time() - t0:.0f} s]")
print(sens.pivot(index="T", columns="regla", values="volatilidad").round(3).to_string())
fig = px.line(sens, x="q", y="volatilidad", color="regla", markers=True, log_x=True, title="Volatilidad fuera de muestra según q = N/T (ventanas de 1.000 a 60 días)")
fig.update_layout(height=400, width=760)
fig.show()



Con ventanas cortas ($q$ cerca de 1) la mínima varianza se descontrola: la matriz que invierte está dominada por ruido
muestral y la inversión amplifica justamente las direcciones peor estimadas. HRP se mantiene estable en todo el rango:
no es la de menor volatilidad cuando hay datos de sobra, pero nunca es la peor, y esa robustez es su argumento
principal.
